In [0]:
%run ./common_utils

In [0]:
CAT = 'machine_health.prod'
AUDIT = CAT + '.pipeline_run_audit'
declare_widgets()
mt = dbutils.widgets.get('machine_type')
with audited() as a:
    hot = (spark.table(f'{CAT}.health_decisions').where(scope(mt)).where('is_healthy = 0')
           .join(spark.table(f'{CAT}.predictions_latest').where('temp_c > 80')
                      .select('sensor_id', 'reading_time', 'axis'),
                 ['sensor_id', 'reading_time', 'axis'])
           .select('sensor_id', 'machine_type', 'reading_time',
                   F.lit('Check cooling').alias('advice')))
    seen = spark.table(f'{CAT}.thermal_advice').select('sensor_id', 'reading_time')
    (hot.join(seen, ['sensor_id', 'reading_time'], 'left_anti')
        .write.mode('append').saveAsTable(f'{CAT}.thermal_advice'))

/home/spark-b21e58f6-d590-4e9b-8d0f-9f/.ipykernel/63/command-7089605854406250-1338218180:94: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  start_time=datetime.utcnow()
/home/spark-b21e58f6-d590-4e9b-8d0f-9f/.ipykernel/63/command-7089605854406250-1338218180:108: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  rec['end_time'] = datetime.utcnow()
